In [1]:
# !pip install tensorflow

In [2]:
import os
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, optimizers, callbacks
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tqdm import tqdm
import matplotlib.pyplot as plt

I0000 00:00:1794638097.303702    7825 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1794638097.304412    7825 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1794638097.334327    7825 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1794638098.020544    7825 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.

In [3]:
DATA_DIR = "./tiny-imagenet-200"
ZIP_URL = "http://cs231n.stanford.edu/tiny-imagenet-200.zip"
ZIP_PATH = "./tiny-imagenet-200.zip"

In [4]:
IMG_SIZE = 224
BATCH_SIZE = 32                 # Reduce to 16 if GPU memory is limited
NUM_CLASSES = 200
EPOCHS = 15
LEARNING_RATE = 1e-4

In [5]:
import urllib.request
import zipfile

if not os.path.exists(DATA_DIR):
    # Check if the zip file already exists locally before downloading
    if not os.path.exists(ZIP_PATH):
        print(f"Downloading Tiny ImageNet-200 from {ZIP_URL}...")
        try:
            urllib.request.urlretrieve(ZIP_URL, ZIP_PATH)
            print("Download completed successfully.")
        except Exception as e:
            print(f"Error downloading dataset: {e}")
            print("Please check your internet connection or the source URL.")
            raise e

    # Extract the file to the current directory
    print(f"Extracting {ZIP_PATH} to directory...")
    try:
        with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
            zip_ref.extractall(".")
        print(f"Dataset extracted and available at: {DATA_DIR}")

        # Optional: Clean up the zip file to free disk space (~248MB)
        os.remove(ZIP_PATH)
        print(f"Removed temporary archive: {ZIP_PATH}")
    except Exception as e:
        print(f"Error during extraction: {e}")
        raise e
else:
    print(f"Dataset already available in directory: {DATA_DIR}")

Dataset already available in directory: ./tiny-imagenet-200


In [6]:
# ============================================================
# Reorganize Validation Set (CRITICAL)
# ============================================================
import shutil

def reorganize_val_folder():
    val_dir = os.path.join(DATA_DIR, "val")
    val_img_dir = os.path.join(val_dir, "images")
    anno_file = os.path.join(val_dir, "val_annotations.txt")

    if not os.path.exists(val_img_dir):
        print("Validation set already reorganized.")
        return

    print("Reorganizing validation set into class folders...")
    with open(anno_file, "r") as f:
        for line in f:
            parts = line.strip().split("\t")
            img_name, class_id = parts[0], parts[1]
            class_folder = os.path.join(val_dir, class_id)
            os.makedirs(class_folder, exist_ok=True)
            src = os.path.join(val_img_dir, img_name)
            dst = os.path.join(class_folder, img_name)
            if os.path.exists(src):
                shutil.move(src, dst)

    # Remove the empty images folder
    if os.path.exists(val_img_dir):
        shutil.rmtree(val_img_dir)
    print("Validation set reorganized successfully.")

reorganize_val_folder()

Validation set already reorganized.


In [7]:
EPS = 8 / 255.0
PGD_STEPS = 7                   # 7–10 is a good balance (higher = stronger but slower)
PGD_STEP_SIZE = 2 / 255.0
ADV_RATIO = 0.5

In [8]:
MODEL_SAVE_PATH = "models/resnet50_adversarial.h5"
os.makedirs("models", exist_ok=True)
os.makedirs("results", exist_ok=True)

In [9]:
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    zoom_range=0.1
)

In [10]:
val_datagen = ImageDataGenerator(rescale=1./255)

In [11]:
train_gen = train_datagen.flow_from_directory(
    os.path.join(DATA_DIR, "train"),
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=True
)

val_gen = val_datagen.flow_from_directory(
    os.path.join(DATA_DIR, "val"),
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False
)

Found 100000 images belonging to 200 classes.
Found 10000 images belonging to 200 classes.


In [12]:
def build_model(trainable_base=False, fine_tune_at=None):
    """
    trainable_base=False  → freeze entire backbone (fast)
    trainable_base=True   → unfreeze entire backbone
    fine_tune_at=100      → freeze layers before layer 100, train the rest (recommended)
    """
    inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))  # model expects [0, 1]

    # Convert [0,1] → [0,255] and apply official ResNet50 preprocessing
    x = layers.Rescaling(255.0)(inputs)
    x = tf.keras.applications.resnet50.preprocess_input(x)

    base = ResNet50(
        weights="imagenet",
        include_top=False,
        input_tensor=x
    )

    if fine_tune_at is not None:
        # Freeze early layers, unfreeze from fine_tune_at onwards
        for layer in base.layers[:fine_tune_at]:
            layer.trainable = False
        for layer in base.layers[fine_tune_at:]:
            layer.trainable = True
    else:
        base.trainable = trainable_base

    x = layers.GlobalAveragePooling2D()(base.output)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)

    model = models.Model(inputs, outputs)
    return model

In [13]:
model = build_model(trainable_base=False)
model.compile(
    optimizer=optimizers.Adam(LEARNING_RATE),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)
optimizer = optimizers.Adam(LEARNING_RATE)

E0000 00:00:1794638099.847581    7825 cuda_executor.cc:1737] INTERNAL: CUDA Runtime error: Failed call to cudaGetRuntimeVersion: Error loading CUDA libraries. GPU will not be used.: Error loading CUDA libraries. GPU will not be used.
W0000 00:00:1794638099.847745    7898 cuda_executor.cc:1755] Failed to determine cuDNN version (Note that this is expected if the application doesn't link the cuDNN plugin): INTERNAL: cuDNN error: CUDNN_STATUS_INTERNAL_ERROR
W0000 00:00:1794638099.870787    7825 gpu_device.cc:2365] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


In [14]:
BASELINE_PATH = "models/resnet50_baseline.h5"

def train_baseline(epochs=12):
    print("\n" + "="*60)
    print("Training BASELINE ResNet-50 (Clean)")
    print("="*60)

    # Stage 1: Train head only
    baseline_model = build_model(trainable_base=False)
    baseline_model.compile(
        optimizer=optimizers.Adam(LEARNING_RATE),
        loss="categorical_crossentropy",
        metrics=["accuracy", "top_k_categorical_accuracy"]
    )

    checkpoint = callbacks.ModelCheckpoint(
        BASELINE_PATH,
        monitor="val_accuracy",
        save_best_only=True,
        verbose=1
    )
    reduce_lr = callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6, verbose=1
    )
    early_stop = callbacks.EarlyStopping(
        monitor="val_accuracy", patience=5, restore_best_weights=True, verbose=1
    )

    print("Stage 1: Training classification head...")
    history1 = baseline_model.fit(
        train_gen,
        epochs=epochs // 2,
        validation_data=val_gen,
        callbacks=[checkpoint, reduce_lr, early_stop],
        verbose=1
    )

    # Stage 2: Fine-tune later layers
    print("\nStage 2: Fine-tuning later layers of ResNet...")
    stage1_best = max(history1.history["val_accuracy"])   # best achieved in stage 1

    baseline_model = build_model(fine_tune_at=140)
    baseline_model.load_weights(BASELINE_PATH)

    baseline_model.compile(
        optimizer=optimizers.Adam(LEARNING_RATE / 10),
        loss="categorical_crossentropy",
        metrics=["accuracy", "top_k_categorical_accuracy"]
    )

    checkpoint2 = callbacks.ModelCheckpoint(
        BASELINE_PATH,
        monitor="val_accuracy",
        save_best_only=True,
        initial_value_threshold=stage1_best,   # only overwrite if stage 2 beats stage 1
        verbose=1
    )
    reduce_lr2 = callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6, verbose=1
    )

    history2 = baseline_model.fit(
        train_gen,
        epochs=epochs // 2,
        validation_data=val_gen,
        callbacks=[checkpoint2, reduce_lr2],
        verbose=1
    )

    print(f"Baseline model saved to: {BASELINE_PATH}")
    return baseline_model, history2

In [15]:
# ============================================================
# 2. GAUSSIAN NOISE EVALUATION
# ============================================================

def add_gaussian_noise_batch(images, sigma, max_val=1.0):
    """
    images: numpy array in [0, max_val]
    sigma: standard deviation relative to max_val
    """
    noise = np.random.normal(0, sigma * max_val, images.shape).astype(np.float32)
    noisy = np.clip(images + noise, 0.0, max_val)
    return noisy


def evaluate_under_noise(model, val_gen, noise_levels=[0.05, 0.10, 0.15, 0.20]):
    print("\n" + "="*60)
    print("Evaluating under Gaussian Noise")
    print("="*60)

    results = {}

    # Clean accuracy
    eval_results = model.evaluate(val_gen, verbose=0)
    acc = eval_results[1]          # accuracy is always the second value
    results["clean"] = acc
    print(f"Clean Accuracy: {acc:.4f}")

    results["clean"] = acc
    print(f"Clean Accuracy: {acc:.4f}")

    for sigma in noise_levels:
        print(f"\nTesting sigma = {sigma} ...")
        correct = 0
        total = 0

        for i in range(len(val_gen)):
            images, labels = val_gen[i]
            noisy_images = add_gaussian_noise_batch(images, sigma)
            preds = model.predict(noisy_images, verbose=0)
            correct += np.sum(np.argmax(preds, axis=1) == np.argmax(labels, axis=1))
            total += len(labels)

        acc = correct / total
        results[f"noise_{sigma}"] = acc
        print(f"Accuracy under σ={sigma}: {acc:.4f}")

    return results

In [16]:
# ============================================================
# 3. NOISE-AUGMENTED TRAINING (Defense against Gaussian Noise)
# ============================================================

NOISE_MODEL_PATH = "models/resnet50_noise_augmented.h5"

def train_noise_augmented(epochs=12, noise_sigma=0.10):
    print("\n" + "="*60)
    print(f"Training Noise-Augmented Model (σ = {noise_sigma})")
    print("="*60)

    # preprocessing_function receives images in [0, 255] (before rescale)
    def noise_preprocessor(x):
        # x is in [0, 255]
        if np.random.rand() < 0.5:
            x = add_gaussian_noise_batch(x, sigma=noise_sigma, max_val=255.0)
        return x

    noise_datagen = ImageDataGenerator(
        rescale=1./255,
        rotation_range=15,
        width_shift_range=0.1,
        height_shift_range=0.1,
        horizontal_flip=True,
        zoom_range=0.1,
        preprocessing_function=noise_preprocessor
    )

    noise_train_gen = noise_datagen.flow_from_directory(
        os.path.join(DATA_DIR, "train"),
        target_size=(IMG_SIZE, IMG_SIZE),
        batch_size=BATCH_SIZE,
        class_mode="categorical",
        shuffle=True
    )

    # Use fine-tuning for better performance
    model_noise = build_model(fine_tune_at=140)   # unfreeze last part of ResNet
    model_noise.compile(
        optimizer=optimizers.Adam(LEARNING_RATE),
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )

    checkpoint = callbacks.ModelCheckpoint(
        NOISE_MODEL_PATH,
        monitor="val_accuracy",
        save_best_only=True,
        verbose=1
    )

    history = model_noise.fit(
        noise_train_gen,
        epochs=epochs,
        validation_data=val_gen,
        callbacks=[checkpoint],
        verbose=1
    )

    print(f"Noise-augmented model saved to: {NOISE_MODEL_PATH}")
    return model_noise, history

In [17]:
@tf.function
def pgd_attack(model, images, labels, eps=EPS, alpha=PGD_STEP_SIZE, steps=PGD_STEPS):
    adv_images = images + tf.random.uniform(tf.shape(images), -eps, eps)
    adv_images = tf.clip_by_value(adv_images, 0.0, 1.0)

    for _ in range(steps):
        with tf.GradientTape() as tape:
            tape.watch(adv_images)
            preds = model(adv_images, training=False)
            loss = tf.keras.losses.categorical_crossentropy(labels, preds)

        gradients = tape.gradient(loss, adv_images)
        adv_images = adv_images + alpha * tf.sign(gradients)
        # Project onto epsilon ball
        adv_images = tf.clip_by_value(adv_images, images - eps, images + eps)
        adv_images = tf.clip_by_value(adv_images, 0.0, 1.0)

    return adv_images

In [18]:
@tf.function
def train_step(model, images, labels, optimizer):
    # Generate adversarial examples
    adv_images = pgd_attack(model, images, labels)

    with tf.GradientTape() as tape:
        # Clean predictions
        clean_preds = model(images, training=True)
        loss_clean = tf.keras.losses.categorical_crossentropy(labels, clean_preds)

        # Adversarial predictions
        adv_preds = model(adv_images, training=True)
        loss_adv = tf.keras.losses.categorical_crossentropy(labels, adv_preds)

        # Combined loss
        loss = (1.0 - ADV_RATIO) * loss_clean + ADV_RATIO * loss_adv
        loss = tf.reduce_mean(loss)

    gradients = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(gradients, model.trainable_variables))

    # Metrics
    clean_acc = tf.reduce_mean(tf.keras.metrics.categorical_accuracy(labels, clean_preds))
    adv_acc = tf.reduce_mean(tf.keras.metrics.categorical_accuracy(labels, adv_preds))

    return loss, clean_acc, adv_acc

In [19]:
def adversarial_train(model, train_gen, val_gen, epochs=EPOCHS):
    history = {
        "loss": [], "clean_acc": [], "adv_acc": [],
        "val_loss": [], "val_acc": []
    }

    best_val_acc = 0.0
    steps_per_epoch = len(train_gen)

    for epoch in range(epochs):
        print(f"\n=== Epoch {epoch+1}/{epochs} ===")
        epoch_loss = []
        epoch_clean_acc = []
        epoch_adv_acc = []

        # Training
        pbar = tqdm(range(steps_per_epoch), desc="Training")
        for step in pbar:
            images, labels = next(train_gen)
            images = tf.convert_to_tensor(images, dtype=tf.float32)
            labels = tf.convert_to_tensor(labels, dtype=tf.float32)

            loss, clean_acc, adv_acc = train_step(model, images, labels, optimizer)

            epoch_loss.append(loss.numpy())
            epoch_clean_acc.append(clean_acc.numpy())
            epoch_adv_acc.append(adv_acc.numpy())

            pbar.set_postfix({
                "loss": f"{np.mean(epoch_loss):.4f}",
                "clean_acc": f"{np.mean(epoch_clean_acc):.4f}",
                "adv_acc": f"{np.mean(epoch_adv_acc):.4f}"
            })

        # Validation (on clean data)
        val_loss, val_acc = model.evaluate(val_gen, verbose=0)

        # Record history
        history["loss"].append(np.mean(epoch_loss))
        history["clean_acc"].append(np.mean(epoch_clean_acc))
        history["adv_acc"].append(np.mean(epoch_adv_acc))
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)

        print(f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")

        # Save best model
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            model.save(MODEL_SAVE_PATH)
            print(f"  → Best model saved (Val Acc = {val_acc:.4f})")

    return history

In [21]:
# ============================================================
# CONTROLLED EXECUTION
# Uncomment only the parts you want to run one at a time
# ============================================================

# ---------- 1. Train Baseline Model ----------
baseline_model, baseline_hist = train_baseline(epochs=12)


Training BASELINE ResNet-50 (Clean)
Stage 1: Training classification head...
Epoch 1/6


I0000 00:00:1794638132.481947    7825 generator_dataset_op.cc:213] Memory patch applied: M_TRIM_THRESHOLD=128 kb was set.


3125/3125 ━━━━━━━━━━━━━━━━━━━━ 0s 503ms/step - accuracy: 0.1503 - loss: 4.2938 - top_k_categorical_accuracy: 0.3357
Epoch 1: val_accuracy improved from None to 0.41080, saving model to models/resnet50_baseline.h5



Epoch 1: finished saving model to models/resnet50_baseline.h5
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 1721s 550ms/step - accuracy: 0.1503 - loss: 4.2938 - top_k_categorical_accuracy: 0.3357 - val_accuracy: 0.4108 - val_loss: 2.5936 - val_top_k_categorical_accuracy: 0.6703 - learning_rate: 1.0000e-04
Epoch 2/6
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 0s 501ms/step - accuracy: 0.3275 - loss: 2.9777 - top_k_categorical_accuracy: 0.5906
Epoch 2: val_accuracy improved from 0.41080 to 0.47530, saving model to models/resnet50_baseline.h5



Epoch 2: finished saving model to models/resnet50_baseline.h5
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 1719s 550ms/step - accuracy: 0.3275 - loss: 2.9777 - top_k_categorical_accuracy: 0.5906 - val_accuracy: 0.4753 - val_loss: 2.2534 - val_top_k_categorical_accuracy: 0.7189 - learning_rate: 1.0000e-04
Epoch 3/6
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 0s 510ms/step - accuracy: 0.3968 - loss: 2.6187 - top_k_categorical_accuracy: 0.6585
Epoch 3: val_accuracy improved from 0.47530 to 0.50230, saving model to models/resnet50_baseline.h5



Epoch 3: finished saving model to models/resnet50_baseline.h5
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 1749s 559ms/step - accuracy: 0.3968 - loss: 2.6187 - top_k_categorical_accuracy: 0.6585 - val_accuracy: 0.5023 - val_loss: 2.1045 - val_top_k_categorical_accuracy: 0.7482 - learning_rate: 1.0000e-04
Epoch 4/6
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 0s 512ms/step - accuracy: 0.4294 - loss: 2.4421 - top_k_categorical_accuracy: 0.6902
Epoch 4: val_accuracy improved from 0.50230 to 0.51980, saving model to models/resnet50_baseline.h5


... (5 outputs left out)


Epoch 3: finished saving model to models/resnet50_baseline.h5
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 2453s 785ms/step - accuracy: 0.6291 - loss: 1.4558 - top_k_categorical_accuracy: 0.8523 - val_accuracy: 0.6616 - val_loss: 1.3364 - val_top_k_categorical_accuracy: 0.8696 - learning_rate: 1.0000e-05
Epoch 4/6
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 0s 739ms/step - accuracy: 0.6523 - loss: 1.3529 - top_k_categorical_accuracy: 0.8672
Epoch 4: val_accuracy improved from 0.66160 to 0.66740, saving model to models/resnet50_baseline.h5



Epoch 4: finished saving model to models/resnet50_baseline.h5
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 2464s 788ms/step - accuracy: 0.6523 - loss: 1.3529 - top_k_categorical_accuracy: 0.8672 - val_accuracy: 0.6674 - val_loss: 1.3017 - val_top_k_categorical_accuracy: 0.8744 - learning_rate: 1.0000e-05
Epoch 5/6
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 0s 744ms/step - accuracy: 0.6737 - loss: 1.2667 - top_k_categorical_accuracy: 0.8777
Epoch 5: val_accuracy improved from 0.66740 to 0.67430, saving model to models/resnet50_baseline.h5



Epoch 5: finished saving model to models/resnet50_baseline.h5
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 2480s 794ms/step - accuracy: 0.6737 - loss: 1.2667 - top_k_categorical_accuracy: 0.8777 - val_accuracy: 0.6743 - val_loss: 1.2851 - val_top_k_categorical_accuracy: 0.8755 - learning_rate: 1.0000e-05
Epoch 6/6
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 0s 726ms/step - accuracy: 0.6913 - loss: 1.1874 - top_k_categorical_accuracy: 0.8896
Epoch 6: val_accuracy improved from 0.67430 to 0.67980, saving model to models/resnet50_baseline.h5



Epoch 6: finished saving model to models/resnet50_baseline.h5
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 2420s 774ms/step - accuracy: 0.6913 - loss: 1.1874 - top_k_categorical_accuracy: 0.8896 - val_accuracy: 0.6798 - val_loss: 1.2551 - val_top_k_categorical_accuracy: 0.8825 - learning_rate: 1.0000e-05
Baseline model saved to: models/resnet50_baseline.h5


In [22]:
# ---------- 2. Evaluate Baseline under Gaussian Noise ----------
baseline_model = tf.keras.models.load_model("models/resnet50_baseline.h5")
noise_results = evaluate_under_noise(baseline_model, val_gen)
print(noise_results)

ValueError: Unknown layer: 'GetItem'. Please ensure you are using a `keras.utils.custom_object_scope` and that this object is included in the scope. See https://www.tensorflow.org/guide/keras/save_and_serialize#registering_the_custom_object for details.

In [ ]:
# ---------- 3. Train Noise-Augmented Model ----------
noise_model, noise_hist = train_noise_augmented(epochs=12, noise_sigma=0.10)

In [ ]:
# ---------- 4. Train Adversarially Robust Model ----------
print("Starting Adversarial Training...")

# Build a FINE-TUNABLE model and start from the trained clean baseline
model = build_model(fine_tune_at=140)        # backbone partially unfrozen
model.load_weights(BASELINE_PATH)            # adversarially fine-tune, don't start from scratch
model.compile(                               # needed because the loop calls model.evaluate()
    optimizer=optimizers.Adam(LEARNING_RATE / 10),
    loss="categorical_crossentropy",
    metrics=["accuracy"]                     # keep exactly 2 outputs — the loop unpacks (val_loss, val_acc)
)

# train_step uses this SAME optimizer object; it must match the model above
optimizer = optimizers.Adam(LEARNING_RATE / 10)

history = adversarial_train(model, train_gen, val_gen, epochs=EPOCHS)

print("\nTraining finished!")
print(f"Best model saved at: {MODEL_SAVE_PATH}")

In [ ]:
# # Optional: Plot training curves
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history["loss"], label="Train Loss")
plt.plot(history["val_loss"], label="Val Loss")
plt.legend()
plt.title("Loss")

plt.subplot(1, 2, 2)
plt.plot(history["clean_acc"], label="Clean Acc")
plt.plot(history["adv_acc"], label="Adv Acc (train)")
plt.plot(history["val_acc"], label="Val Acc")
plt.legend()
plt.title("Accuracy")
plt.tight_layout()
plt.savefig("results/adversarial_training_curves.png", dpi=150)
plt.show()